# Transformer Foundations, Part 1: Tokenization and Embeddings

> **Guiding question:** Before a Transformer can move information between tokens, how does text become stable IDs and trainable vectors?

## 0. The Challenge: Models Cannot Read Strings

Start with one sentence:

```text
the cat sat on the mat
```

A model needs numbers, but two conversions solve different problems:

```mermaid
flowchart LR
    A["Raw text"] --> B["Tokenizer"]
    B --> C["Tokens"]
    C --> D["Vocabulary IDs"]
    D --> E["Embedding lookup"]
    E --> F["One trainable vector per position"]
    F --> G["Prediction loss"]
    G --> H["Backprop updates used rows"]
```

- The **tokenizer** decides the pieces and assigns each piece a stable ID.
- The **embedder** uses each ID as a row address in a trainable table.

This chapter ends at the failure that creates the need for the next one: embeddings preserve token identity, but the same words in a different order still collapse under order-blind pooling.

| Step | Visible question | Evidence |
|---|---|---|
| Tokenize | What pieces does the model receive? | Word and tiny subword traces |
| Assign IDs | Does each piece keep one stable address? | Encode/decode round trip |
| Embed | What does an ID retrieve? | Table lookup and repeated-token equality |
| Learn | Which rows receive responsibility? | Gradient and update norms by row |
| Expose the gap | Do embeddings alone preserve order? | Shuffled mean vectors are identical |

## Prerequisite Bridge

The broader tokenizer survey lives in [GenAI Prerequisite 06](../../genai-prerequisites/06-tokenization/tokenization-and-embeddings.ipynb). This chapter is the Transformer-specific bridge: it keeps the path from text to lookup rows to prediction feedback visible in one small example.

You should already be comfortable with tensors, `nn.Module`, raw logits, cross-entropy loss, and `.backward()` from the [PyTorch fundamentals](../../genai-prerequisites/03-pytorch-fundamentals/01-keras-to-pytorch-antarctic-field-guide.ipynb).

**Scope:** the tiny greedy subword tokenizer below exposes the contract. Production BPE, WordPiece, and Unigram tokenizers learn their pieces from a corpus and handle normalization, byte fallbacks, and special-token policy more carefully.

In [ ]:
# Imports and deterministic state
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white"})
print(f"torch={torch.__version__}; seed={SEED}")

---

## 1. Tokenization: Choose the Pieces Before Choosing the Numbers

A tokenizer is not an embedder. It first answers: **what counts as one piece?**

A whole-word tokenizer is readable, but an unseen word becomes `<UNK>`. A subword tokenizer can reuse smaller known pieces:

```text
"kitten" -> whole word: <UNK>
"kitten" -> tiny subwords: kit + ten
```

The code uses longest-match splitting only to make that fallback inspectable. It is not a hand-written replacement for a production tokenizer.

**Predict:** Will the two occurrences of `the` receive the same ID? Will `kitten` require a new full-word vocabulary entry?

In [ ]:
# A tiny inspectable vocabulary and greedy subword tokenizer
VOCAB_PIECES = [
    "<PAD>", "<BOS>", "<EOS>", "<UNK>",
    "the", "cat", "sat", "on", "mat", "dog", "rug", "kit", "ten", ".",
]
VOCAB = {piece: index for index, piece in enumerate(VOCAB_PIECES)}
ID_TO_TOKEN = {index: piece for piece, index in VOCAB.items()}
SPECIAL_TOKENS = {"<PAD>", "<BOS>", "<EOS>", "<UNK>"}
SEARCH_PIECES = sorted(
    [piece for piece in VOCAB if piece not in SPECIAL_TOKENS and piece != "."],
    key=len,
    reverse=True,
)


def split_word(word: str) -> list[str]:
    if word in VOCAB:
        return [word]
    pieces, cursor = [], 0
    while cursor < len(word):
        match = next((piece for piece in SEARCH_PIECES if word.startswith(piece, cursor)), None)
        if match is None:
            return ["<UNK>"]
        pieces.append(match)
        cursor += len(match)
    return pieces


def tokenize(text: str) -> list[str]:
    words_and_punctuation = re.findall(r"[a-z]+|[.]", text.lower())
    return [piece for item in words_and_punctuation for piece in split_word(item)]


def encode(text: str, add_bos: bool = False, add_eos: bool = False) -> list[int]:
    pieces = tokenize(text)
    if add_bos:
        pieces.insert(0, "<BOS>")
    if add_eos:
        pieces.append("<EOS>")
    return [VOCAB.get(piece, VOCAB["<UNK>"]) for piece in pieces]


def decode(token_ids: list[int]) -> str:
    return " ".join(ID_TO_TOKEN[token_id] for token_id in token_ids)


for text in ["the cat sat on the mat", "the kitten sat on the mat", "the fox sat"]:
    pieces = tokenize(text)
    token_ids = encode(text)
    print(f"{text!r}\n  pieces={pieces}\n  ids={token_ids}\n  decoded={decode(token_ids)!r}")

sentence_ids = encode("the cat sat on the mat")
assert sentence_ids[0] == sentence_ids[4] == VOCAB["the"]
assert tokenize("kitten") == ["kit", "ten"]
print("PASS: repeated pieces keep one ID; the known subwords compose 'kitten'.")

#### What the tokenizer fixed, and what it did not

The same piece now maps to the same integer everywhere. That stability lets saved model weights agree on what every row means.

But an ID is only an address. ID `5` is not more meaningful than ID `4`, and arithmetic on IDs does not create semantics. We need a table whose rows can learn useful coordinates.

---

## 2. Embeddings: IDs Address Trainable Rows

An embedding table has one row per vocabulary item. Looking up six token IDs returns six vectors:

```text
token IDs                  (sequence)
embedding table lookup     (sequence, model width)
```

Two occurrences of `the` retrieve the same row before context is added. Their vectors can diverge later only after position and surrounding tokens enter the computation.

In [ ]:
# Trainable embedding lookup
D_EMBED = 8
embedder = nn.Embedding(len(VOCAB), D_EMBED, padding_idx=VOCAB["<PAD>"])
token_ids = torch.tensor(sentence_ids, dtype=torch.long)
token_vectors = embedder(token_ids)
tokens = tokenize("the cat sat on the mat")

assert token_vectors.shape == (len(tokens), D_EMBED)
assert torch.allclose(token_vectors[0], token_vectors[4])

fig, axis = plt.subplots(figsize=(9, 3.6))
sns.heatmap(token_vectors.detach().numpy(), cmap="RdBu_r", center=0, ax=axis,
            xticklabels=[f"d{index}" for index in range(D_EMBED)], yticklabels=tokens)
axis.set_title("One embedding row retrieved at each token position")
axis.set_xlabel("Learned feature coordinate"); axis.set_ylabel("Token position")
plt.tight_layout(); plt.show()

print(f"IDs shape={tuple(token_ids.shape)} -> vectors shape={tuple(token_vectors.shape)}")
print(f"Repeated 'the' vectors equal before context: {torch.allclose(token_vectors[0], token_vectors[4])}")

## 3. Prediction Error Teaches the Rows

Embedding coordinates are not hand-written meanings. They start randomly and become useful only because prediction loss sends gradients backward.

For one tiny lesson, the visible prefix is `the cat sat on the` and the target is `mat`. We deliberately mean-pool the prefix so the learning path stays visible:

```text
prefix IDs -> embedding rows -> pooled vector -> vocabulary logits
-> loss against "mat" -> gradients -> optimizer update
```

**Predict:** Which embedding rows should move: every vocabulary row, or only rows used by the prefix?

In [ ]:
# One measured backward pass through the embedder
torch.manual_seed(SEED)
embedder = nn.Embedding(len(VOCAB), D_EMBED, padding_idx=VOCAB["<PAD>"])
output_head = nn.Linear(D_EMBED, len(VOCAB))
optimizer = torch.optim.SGD([*embedder.parameters(), *output_head.parameters()], lr=0.25)

prefix_ids = torch.tensor(encode("the cat sat on the"), dtype=torch.long)
target_id = torch.tensor([VOCAB["mat"]], dtype=torch.long)
before = embedder.weight.detach().clone()

optimizer.zero_grad()
context = embedder(prefix_ids).mean(dim=0, keepdim=True)
logits = output_head(context)
loss = F.cross_entropy(logits, target_id)
loss.backward()

gradient_norms = embedder.weight.grad.norm(dim=1)
optimizer.step()
update_norms = (embedder.weight.detach() - before).norm(dim=1)
used_ids = set(prefix_ids.tolist())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharex=True)
axes[0].bar(VOCAB_PIECES, gradient_norms.numpy(), color="#2a6f97")
axes[0].set_title("Embedding gradient by vocabulary row"); axes[0].set_ylabel("gradient norm")
axes[1].bar(VOCAB_PIECES, update_norms.numpy(), color="#d97706")
axes[1].set_title("Actual row movement after one update"); axes[1].set_ylabel("update norm")
for axis in axes:
    axis.tick_params(axis="x", rotation=55)
plt.tight_layout(); plt.show()

assert all(gradient_norms[index] > 0 for index in used_ids)
assert all(gradient_norms[index] == 0 for index in range(len(VOCAB)) if index not in used_ids)
print(f"loss={loss.item():.4f}; used rows={[ID_TO_TOKEN[index] for index in sorted(used_ids)]}")
print("PASS: prediction error reached every used embedding row and no unused row.")

#### What was learned

The output head made a next-token guess, loss measured the error, and backpropagation assigned responsibility to the lookup rows that produced the prefix representation. A real language model replaces mean pooling with many Transformer blocks, but the embedding rows still learn through this same backward path.

This tiny update does not prove that a row acquired a stable human concept. It proves the training contract: used rows are differentiable parameters connected to prediction error.

---

## 4. The Ordering Failure

Embeddings solve token identity, not token order. Compare:

```text
the cat sat on the mat
the mat sat on the cat
```

Both sequences contain exactly the same IDs. Any representation that simply sums or averages their embedding rows must be identical.

**Predict:** Can an order-blind pooled vector distinguish who sat where?

In [ ]:
# Same embedding rows, different order, identical mean
def mean_embedding(text: str) -> torch.Tensor:
    ids = torch.tensor(encode(text), dtype=torch.long)
    return embedder(ids).mean(dim=0)


ordered = "the cat sat on the mat"
reversed_roles = "the mat sat on the cat"
ordered_mean = mean_embedding(ordered)
reversed_mean = mean_embedding(reversed_roles)
maximum_difference = (ordered_mean - reversed_mean).abs().max().item()

assert torch.allclose(ordered_mean, reversed_mean, atol=1e-7)
print(f"Maximum pooled-vector difference: {maximum_difference:.8f}")
print("PASS: embeddings preserve which tokens appeared, but order-blind pooling loses where they appeared.")

---

## Chapter 1 Checkpoint

```text
text -> token pieces -> stable IDs -> trainable embedding rows
     -> prediction loss -> gradients update used rows
```

| Question | Measured answer |
|---|---|
| Can an unseen whole word reuse known pieces? | `kitten -> kit + ten` |
| Do repeated pieces keep one address? | both `the` positions used the same ID |
| Do repeated IDs retrieve one row? | their pre-context vectors were equal |
| Does prediction error reach embeddings? | every used row had non-zero gradient and movement |
| Do embeddings alone preserve order? | no; shuffled mean vectors were identical |

**Your turn:** Replace `cat`/`mat` with `dog`/`rug`, predict which embedding rows receive gradients, then verify that swapping word order still leaves mean pooling unchanged.

**Next:** [Part 2 — Attention and Position](02-attention-and-position.ipynb) gives tokens direct access to one another, exposes attention's order blindness, and then introduces additive position and RoPE.